# Simulador Cache 4-Way no Colab

Este notebook executa o projeto público [Simulador-Cache-4-Way](https://github.com/LucasMGcode/Simulador-Cache-4-Way) no Google Colab.

A ideia é clonar o repositório, instalar o Icarus Verilog e rodar o testbench auto-verificável com `make sim`.

**English summary:** this notebook clones the public repository, installs Icarus Verilog, and runs the self-checking 4-way cache simulation.


## Material de apoio

- [README do projeto](https://github.com/LucasMGcode/Simulador-Cache-4-Way)
- [SVG do datapath](https://github.com/LucasMGcode/Simulador-Cache-4-Way/blob/main/assets/cache4way_datapath.svg)
- [Datapath da cache](https://github.com/LucasMGcode/Simulador-Cache-4-Way/blob/main/docs/datapath.md)
- [Desenho Inkscape e marcadores](https://github.com/LucasMGcode/Simulador-Cache-4-Way/blob/main/docs/inkscape-drawing.md)
- [FSM principal](https://github.com/LucasMGcode/Simulador-Cache-4-Way/blob/main/docs/fsm.md)
- [Política LRU](https://github.com/LucasMGcode/Simulador-Cache-4-Way/blob/main/docs/lru.md)
- [Roteiro de apresentação](https://github.com/LucasMGcode/Simulador-Cache-4-Way/blob/main/docs/presentation.md)


In [ ]:
#@title 1. Instalar Icarus Verilog
# Instala o Icarus Verilog no ambiente do Colab.
!apt-get update -qq
!apt-get install -y -qq iverilog
!iverilog -V | head -n 3
print("Icarus Verilog instalado com sucesso.")


In [ ]:
#@title 2. Clonar repositório
# Clona a versão pública mais recente do projeto.
!rm -rf Simulador-Cache-4-Way
!git clone https://github.com/LucasMGcode/Simulador-Cache-4-Way.git
%cd Simulador-Cache-4-Way
!git log --oneline -1
print("Repositório clonado com sucesso.")


In [ ]:
#@title 3. Executar simulação
# Executa a simulação e o testbench auto-verificável.
import subprocess

%cd /content/Simulador-Cache-4-Way/src
subprocess.run(["make", "sim"], check=True)
print("Simulação executada com sucesso.")


## Visualização do datapath

A próxima célula exibe o SVG autoral do datapath. Ele é editável no Inkscape e contém marcadores `@...` que serão substituídos com os dados gerados pela simulação.

**English summary:** the next cells render the Inkscape-editable SVG and fill its dynamic markers using the simulation trace.


In [ ]:
#@title 4. Exibir desenho estático do datapath
from pathlib import Path
from IPython.display import HTML, display

REPO = Path("/content/Simulador-Cache-4-Way")
SVG_PATH = REPO / "assets" / "cache4way_datapath.svg"

def display_responsive_svg(svg_text):
    display(HTML(f"""
    <div style="width:100%; overflow-x:auto; padding: 8px 0;">
      <div style="min-width: 1120px; max-width: 1600px; margin: 0 auto;">
        {svg_text.replace('<svg', '<svg style="width:100%; height:auto; display:block;"', 1)}
      </div>
    </div>
    """))

display_responsive_svg(SVG_PATH.read_text(encoding="utf-8"))


## Visualizações dinâmicas sincronizadas

O explorador abaixo controla duas leituras complementares do mesmo acesso:

- **Datapath:** mostra os sinais locais do acesso atual, o caminho de decisão e a saída.
- **Matriz de estado:** mostra o estado global da cache inteira após o acesso, em `4 conjuntos x 4 vias`.

Use `Anterior`, `Próximo` ou o seletor de passo para navegar pela sequência de acessos.

**English summary:** the same step selector drives a local datapath view and a full-cache grid view after each completed access.


In [ ]:
#@title 5. Preparar visualizações interativas
import csv
import html
import re
from collections import defaultdict
from pathlib import Path

import ipywidgets as widgets
from IPython.display import HTML, clear_output, display

REPO = Path("/content/Simulador-Cache-4-Way")
SVG_PATH = REPO / "assets" / "cache4way_datapath.svg"
TRACE_PATH = REPO / "src" / "trace.csv"
TRACE_GRID_PATH = REPO / "src" / "trace_grid.csv"

template_svg = SVG_PATH.read_text(encoding="utf-8")
rows = list(csv.DictReader(TRACE_PATH.open(encoding="utf-8")))
grid_rows = list(csv.DictReader(TRACE_GRID_PATH.open(encoding="utf-8")))
grid_rows_by_step = defaultdict(list)
for grid_row in grid_rows:
    grid_rows_by_step[grid_row["step"]].append(grid_row)

STATE_NAMES = {
    "0": "COMPARE",
    "1": "HIT",
    "2": "MISS",
    "3": "FILL_BLOCK",
    "4": "UPDATE_TAG",
}

WAY_COLORS = {
    "0": ("#4cc9f0", "rgba(14,165,233,.16)"),
    "1": ("#d8b4fe", "rgba(168,85,247,.16)"),
    "2": ("#fde047", "rgba(234,179,8,.16)"),
    "3": ("#86efac", "rgba(34,197,94,.16)"),
}

def event_class(event):
    if event == "hit" or event.startswith("hit-"):
        return "hit"
    if "replace" in event:
        return "replace"
    return "miss"

def responsive_svg_html(svg_text):
    svg_text = svg_text.replace('<svg', '<svg style="width:100%; height:auto; display:block;"', 1)
    return f"""
    <div style="width:100%; overflow-x:auto; padding: 8px 0 4px;">
      <div style="min-width: 1120px; max-width: 1600px; margin: 0 auto;">
        {svg_text}
      </div>
    </div>
    """

def render_svg(row):
    addr = int(row["addr"])
    tag = int(row["tag"])
    line = int(row["line"])
    blk = int(row["blk"])
    replacements = {
        "@addr": row["addr"],
        "@addr_bin": f"{addr:012b}",
        "@tag": row["tag"],
        "@tag_bin": f"{tag:08b}",
        "@line": row["line"],
        "@line_bin": f"{line:02b}",
        "@blk": row["blk"],
        "@blk_bin": f"{blk:02b}",
        "@hit": row["hit"],
        "@selected_way": row["selected_way"],
        "@dout": row["dout"],
        "@state": STATE_NAMES.get(row["state"], row["state"]),
        "@event": row["event"],
    }
    for i in range(4):
        replacements[f"@valid{i}"] = row[f"valid{i}"]
        replacements[f"@tag{i}"] = row[f"tag{i}"]
        replacements[f"@lru{i}"] = row[f"lru{i}"]

    svg = template_svg
    for marker, value in sorted(replacements.items(), key=lambda item: len(item[0]), reverse=True):
        svg = svg.replace(marker, html.escape(str(value)))

    selected = row["selected_way"]
    svg = svg.replace(
        f'id="way{selected}-card" class="way-card"',
        f'id="way{selected}-card" class="way-card selected"',
    )
    svg = svg.replace(
        'id="event-badge" class="event-badge"',
        f'id="event-badge" class="event-badge {event_class(row["event"])}"',
    )
    remaining = sorted(set(re.findall(r"@\w+", svg)))
    if remaining:
        raise ValueError(f"Marcadores sem substituição: {remaining}")
    return svg

def row_log(row):
    state = STATE_NAMES.get(row["state"], row["state"])
    cells = [
        ("Passo", row["step"]),
        ("Cenário", row["label"]),
        ("Evento", row["event"]),
        ("Endereço", row["addr"]),
        ("Tag / Line / Blk", f'{row["tag"]} / {row["line"]} / {row["blk"]}'),
        ("Hit", row["hit"]),
        ("Via", row["selected_way"]),
        ("Dout", row["dout"]),
        ("FSM", state),
        ("LRU", f'{row["lru0"]}, {row["lru1"]}, {row["lru2"]}, {row["lru3"]}'),
    ]
    cards = "".join(
        f"""<div style="background:rgba(255,255,255,.82); border:1px solid #dbe4ee;
                    border-radius:14px; padding:10px 12px; box-shadow:0 8px 20px rgba(15,23,42,.08);
                    min-width:0; height:58px; overflow:hidden; box-sizing:border-box;">
              <div style="font:600 11px Georgia,serif; color:#64748b; height:14px; line-height:14px;">{html.escape(name)}</div>
              <div style="font:800 15px ui-monospace,Menlo,Consolas,monospace; color:#0f172a;
                          line-height:18px; max-height:36px; white-space:normal; overflow:hidden;
                          overflow-wrap:break-word; word-break:normal;">{html.escape(str(value))}</div>
            </div>"""
        for name, value in cells
    )
    return f"""
    <div style="max-width:1600px; margin:10px auto 0; padding:14px; border-radius:20px;
                background:linear-gradient(135deg, rgba(248,251,255,.92), rgba(239,246,255,.72));
                border:1px solid #dbe4ee; box-shadow:0 12px 28px rgba(15,23,42,.10);">
      <div style="font:800 16px Georgia,serif; color:#1e293b; margin-bottom:10px;">Log do acesso selecionado</div>
      <div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(170px,1fr)); gap:10px;">{cards}</div>
    </div>
    """

def cache_grid_html(row, snapshot):
    active_set = row["line"]
    active_way = row["selected_way"]
    by_position = {(cell["set"], cell["way"]): cell for cell in snapshot}
    way_headers = "".join(
        f'<div class="cg-head cg-way{way}">Via {way}</div>'
        for way in range(4)
    )
    body_rows = []
    for set_idx in range(4):
        row_active = str(set_idx) == active_set
        cells = [f'<div class="cg-set {"active" if row_active else ""}">Conj. {set_idx}</div>']
        for way_idx in range(4):
            cell = by_position[(str(set_idx), str(way_idx))]
            valid = cell["valid"] == "1"
            selected = row_active and str(way_idx) == active_way
            tag = f'0x{int(cell["tag"]):02X}'
            lru = cell["lru"] if valid else "-"
            color, tint = WAY_COLORS[str(way_idx)]
            classes = ["cg-card", f"way{way_idx}"]
            if not valid:
                classes.append("invalid")
            if selected:
                classes.append("selected")
            cells.append(
                f"""<div class="{' '.join(classes)}" style="--way-color:{color}; --way-tint:{tint};">
                      <div class="cg-tag">Tag: {tag}</div>
                      <div class="cg-badges">
                        <span class="cg-valid {'on' if valid else 'off'}">V: {cell["valid"]}</span>
                        <span class="cg-lru">LRU: {lru}</span>
                      </div>
                    </div>"""
            )
        body_rows.append('<div class="cg-row">' + ''.join(cells) + '</div>')

    return f"""
    <style>
      .cg-shell {{
        max-width:1600px; margin:18px auto 0; padding:24px; border-radius:24px;
        background:radial-gradient(circle at top left, rgba(14,165,233,.12), transparent 34%),
                   linear-gradient(145deg, #07111f, #081526 52%, #0a1830);
        border:1px solid rgba(96,165,250,.26); color:#cbd5e1;
        box-shadow:0 18px 40px rgba(2,6,23,.38), inset 0 1px 0 rgba(148,163,184,.14);
      }}
      .cg-top {{ display:flex; justify-content:space-between; gap:20px; align-items:flex-start; margin-bottom:18px; }}
      .cg-title {{ font:800 24px ui-monospace,Menlo,Consolas,monospace; color:#7dd3fc; letter-spacing:.03em; }}
      .cg-subtitle {{ margin-top:6px; color:#94a3b8; font:500 13px ui-monospace,Menlo,Consolas,monospace; }}
      .cg-config {{ min-width:250px; padding:14px 16px; border-radius:16px; background:rgba(15,23,42,.74);
                    border:1px solid rgba(125,211,252,.22); font:600 12px ui-monospace,Menlo,Consolas,monospace; }}
      .cg-config b {{ color:#7dd3fc; }}
      .cg-grid {{ overflow-x:auto; }}
      .cg-header, .cg-row {{ min-width:860px; display:grid; grid-template-columns:140px repeat(4, 1fr); gap:10px; }}
      .cg-header {{ margin-bottom:10px; }}
      .cg-head {{ padding:12px 14px; border-bottom:1px solid rgba(148,163,184,.24);
                  font:800 14px ui-monospace,Menlo,Consolas,monospace; }}
      .cg-way0 {{ color:#4cc9f0; }} .cg-way1 {{ color:#d8b4fe; }}
      .cg-way2 {{ color:#fde047; }} .cg-way3 {{ color:#86efac; }}
      .cg-row {{ margin-bottom:10px; align-items:stretch; }}
      .cg-set {{ display:flex; align-items:center; padding:0 14px; border-radius:14px;
                 background:rgba(15,23,42,.42); color:#93c5fd; font:800 14px ui-monospace,Menlo,Consolas,monospace; }}
      .cg-set.active {{ color:#e0f2fe; border:1px solid rgba(125,211,252,.55); box-shadow:inset 0 0 0 1px rgba(125,211,252,.12); }}
      .cg-card {{ min-height:74px; padding:12px 14px; border-radius:16px; border:1px solid var(--way-color);
                  background:linear-gradient(145deg, var(--way-tint), rgba(15,23,42,.72));
                  box-shadow:inset 0 1px 0 rgba(255,255,255,.07); }}
      .cg-card.invalid {{ border-color:rgba(148,163,184,.34); background:rgba(15,23,42,.42); filter:saturate(.42); }}
      .cg-card.selected {{ box-shadow:0 0 0 2px rgba(125,211,252,.92), 0 0 28px rgba(56,189,248,.26), inset 0 1px 0 rgba(255,255,255,.10); }}
      .cg-tag {{ color:#e2e8f0; font:800 15px ui-monospace,Menlo,Consolas,monospace; }}
      .cg-badges {{ display:flex; gap:8px; margin-top:10px; }}
      .cg-valid, .cg-lru {{ padding:4px 9px; border-radius:8px; font:800 12px ui-monospace,Menlo,Consolas,monospace; }}
      .cg-valid.on {{ color:#86efac; background:rgba(22,101,52,.34); border:1px solid rgba(74,222,128,.28); }}
      .cg-valid.off {{ color:#fca5a5; background:rgba(127,29,29,.30); border:1px solid rgba(248,113,113,.24); }}
      .cg-lru {{ color:#bae6fd; background:rgba(14,116,144,.22); border:1px solid rgba(56,189,248,.22); }}
      .cg-footer {{ display:flex; gap:12px; flex-wrap:wrap; align-items:center; margin-top:16px; color:#94a3b8;
                    font:600 12px ui-monospace,Menlo,Consolas,monospace; }}
      .cg-pill {{ padding:7px 10px; border-radius:999px; border:1px solid rgba(148,163,184,.20); background:rgba(15,23,42,.54); }}
    </style>
    <section class="cg-shell">
      <div class="cg-top">
        <div>
          <div class="cg-title">Matriz de Estado da Cache (Grid View)</div>
          <div class="cg-subtitle">Estado global após o passo {html.escape(row["step"])}: {html.escape(row["label"])}</div>
        </div>
        <div class="cg-config">
          Configuração: <b>Cache 4-Way</b><br>
          Conjuntos (Sets): <b>4</b><br>
          Vias (Ways): <b>4</b><br>
          Política: <b>LRU</b>
        </div>
      </div>
      <div class="cg-grid">
        <div class="cg-header"><div class="cg-head">Conjunto</div>{way_headers}</div>
        {''.join(body_rows)}
      </div>
      <div class="cg-footer">
        <span class="cg-pill">V:1 = válido</span>
        <span class="cg-pill">V:0 = inválido</span>
        <span class="cg-pill">LRU 0 = via mais recente</span>
        <span class="cg-pill">LRU 3 = via menos recente</span>
        <span class="cg-pill">Conjunto ativo: {html.escape(active_set)}</span>
        <span class="cg-pill">Via selecionada: {html.escape(active_way)}</span>
      </div>
    </section>
    """

slider = widgets.IntSlider(
    value=1,
    min=1,
    max=len(rows),
    step=1,
    description="Passo:",
    layout=widgets.Layout(width="420px"),
)
prev_button = widgets.Button(description="Anterior", layout=widgets.Layout(width="110px"))
next_button = widgets.Button(description="Próximo", layout=widgets.Layout(width="110px"))
svg_output = widgets.Output()
log_output = widgets.Output()
grid_output = widgets.Output()

def show_step(step):
    row = rows[step - 1]
    snapshot = grid_rows_by_step[row["step"]]
    with svg_output:
        clear_output(wait=True)
        display(HTML(responsive_svg_html(render_svg(row))))
    with log_output:
        clear_output(wait=True)
        display(HTML(row_log(row)))
    with grid_output:
        clear_output(wait=True)
        display(HTML(cache_grid_html(row, snapshot)))

def on_slider_change(change):
    if change["name"] == "value":
        show_step(change["new"])

def previous(_):
    slider.value = max(slider.min, slider.value - 1)

def next_(_):
    slider.value = min(slider.max, slider.value + 1)

prev_button.on_click(previous)
next_button.on_click(next_)
slider.observe(on_slider_change, names="value")

controls = widgets.HBox(
    [prev_button, next_button, slider],
    layout=widgets.Layout(margin="10px 0 6px"),
)


### Controles do explorador

O mesmo passo selecionado atualiza as três visualizações abaixo.


In [ ]:
#@title 6. Exibir controles do explorador
display(controls)
show_step(slider.value)


### Datapath do acesso selecionado


In [ ]:
#@title 7. Exibir datapath dinâmico
display(svg_output)


### Log do acesso selecionado


In [ ]:
#@title 8. Exibir log do acesso
display(log_output)


### Matriz global da cache


In [ ]:
#@title 9. Exibir matriz global da cache
display(grid_output)


## Como interpretar a saída

O testbench imprime cada acesso à cache, valida os sinais esperados e gera dois rastros para a visualização.

- `ACCESS`: mostra o cenário testado, endereço, tag, linha, bloco, hit, via selecionada, dado e LRU.
- `PASS`: indica que o valor observado bateu com o valor esperado.
- `FAIL`: indica erro; o testbench chama `$fatal(1)` e a simulação termina com falha.
- `ALL TESTS PASSED`: indica que todos os cenários principais passaram.
- `trace.csv`: alimenta o SVG do datapath, substituindo os marcadores `@...` por sinais do acesso selecionado.
- `trace_grid.csv`: alimenta a matriz global, com um snapshot pós-acesso de todos os `4 conjuntos x 4 vias`.

### Campos do log

- `Passo`: posição do acesso na sequência simulada.
- `Cenário`: nome do caso exercitado pelo testbench.
- `Evento`: classe do acesso, como `hit`, `miss-fill` ou `miss-replace`.
- `Endereço`: endereço solicitado à cache.
- `Tag / Line / Blk`: decomposição do endereço em tag, conjunto e offset de bloco.
- `Hit`: indica se o bloco já estava presente na cache.
- `Via`: via usada para o hit ou escolhida para preenchimento/substituição.
- `Dout`: dado devolvido pela cache.
- `FSM`: estado final observado na máquina de estados.
- `LRU`: idades das quatro vias, em que `0` é a mais recente e `3` a menos recente.

A leitura das duas visualizações é complementar: o datapath mostra **como** um acesso é processado; a matriz mostra **como a cache inteira ficou** depois desse acesso.

Cenários cobertos: miss com via inválida, hit após carregamento, substituição com todas as vias válidas, atualização da política LRU, acessos a diferentes linhas da cache e offsets diferentes dentro do mesmo bloco.
